<div style="display: flex; align-items: center;">
  <img src="../Images/SemanticKernelLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">Workshop (AI Extensions) - Premortem (What-If) Decision Analysis  </span>
</div>

> 📜 **_"The pre-mortem method has a number of benefits besides identifying trouble spots in a plan – reducing overconfidence and promoting discoveries. The method can strengthen members’ mental models… And it can help create a culture of candor and trust."_**
>
> -- <cite>Gary Klein (Psychologist, creator of the Pre-mortem method)</cite>  

Decision Intelligence applied in this module:  
* Using Premortem Decision Analysis to improve decision quality & outcomes  
* Using a Premortem decision output for improving of Gathering Intelligence 
* Attain different perspectives using mutliple decision frames  
* Apply a Decision Framework to Premortem Decision Analysis 


### What is a Premortem (What-If Analysis)  

You have probably heard of a **postmortem (post-mortem)**, which is a look back at an actual failure (or success) to understand why it happened and learn from it. In the context of decision making, after the decision outcome is clear you can analyze why the result occured. For example, a hedge fund may host a postmortem meeting after analyzing an unsuccesful move on a tech stock that didn't materialize the return on investment as was planned. 

In a **premortem (pre-mortem)** the outcome is "imagined" and then it is figured out why that potential "imagined" outcome happened. In a premortem, a planning exercise is held where a team imagines that a project or decision has already succeeded or failed. The team then works backward, brainstorming what might have caused things to go wrong. This approach allows everyone to spot potential risks and address them before starting the project. Therefore, it’s a bit like predicting the future and using that prediction to make the forecast stronger. 

### Example Decision Scenario - Purchasing a Car   

Imagine a decision scenario, where you are thinking about buying a new car. Let's play a game of "What-If" that the purchase decision results in either two outcomes a: **disaster** or a **successful** outcome. 

#### Premortem Outcome - Disaster  

<img style="display: block; margin: auto;" width ="700px" src="../Images/Scenarios/Scenario-DecisionPremortem.png">

In a **premortem (pre-mortem)**, you pretend the car purchase turned out to be a disaster, which resulted in a highly undesirable decision outcome. Now, you ask yourself:  
**What went wrong with the decision to buy a new car?**

Here are some potential reasons the car purchase had a poor outcome:  

* **Financial issues:** You realize later you can’t comfortably afford the monthly payments or maintenance costs.  
* **Unexpected repairs:** The car has hidden mechanical problems, leading to costly repairs.  
* **Insurance costs:** Insurance is much higher than expected, impacting your budget.
* **Poor fuel efficiency:** The car uses more gas than you thought, making it expensive to drive.
Resale value drop: The car’s value plummets quickly, leaving you with less if you want to sell or trade it.  

#### Premortem Outcome - Success

In a premortem for a successful car purchase, imagine you bought the car, and it turned out to be a succssful decision. Now, you ask yourself:  
**What went right with the decision to buy a new car?**  

Here are some potential reasons the car purchase had a good outcome: 

* **Affordable and realistic budget:** You set a budget that allowed for comfortable monthly payments and covered insurance, fuel, and maintenance without straining your finances.  
* **Thorough inspection:** You got a detailed inspection before buying, which confirmed the car was in excellent condition and minimized the risk of future repairs.
* **Reliable model choice:** You researched models with a reputation for reliability, so you experience fewer repairs and smooth performance.
* **Fuel efficiency matches needs:** You chose a car that meets your fuel efficiency needs, saving money on gas and fitting well with your daily driving habits.   
* **High resale value:** You picked a make and model that holds its value well, giving you confidence that it’s a good long-term investment.  

#### Premortem (What-If) Decision Analysis 

Imagining successful outcomes helps you focus on actions like sticking to a car budget, getting a full inspection, and choosing a reliable model. You will be more likely to make a purchase you’ll be happy with. By plyaing a "game" of What-If with decision criteria in advance, you could adapt your strategy. For example, you could consider to thoroughly check maintenance records, buy additional warranty, test drive with similar terrain in your area, or reassess your budget to avoid these potential problems.

How can Generative AI help? Using Generative AI, the LLM models can be used to "imagine" premortem outcomes and  brainstorm potential factors why that certain outcome occured. Generative AI agents can act as your "premortem" team. This can lead to additional questions, gathering of intelligence, checklists, negotiation leverage etc. you didn't consider initially. 

---
### Step 1 - Initialize Configuration Builder & Build the AI Orchestration

In [6]:
// Import the required NuGet configuration packages
#r "nuget: Microsoft.Extensions.Configuration, 10.0.12"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.12"
#r "nuget: System.Text.Json, 10.0.12"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

// Load the configuration settings from the local.settings.json and secrets.settings.json files
// The secrets.settings.json file is used to store sensitive information such as API keys
var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];

Installed Packages Microsoft.Extensions.Configuration, 10.0.12 Microsoft.Extensions.Configuration.Json, 10.0.12 System.Text.Json, 10.0.12

In [7]:
// Import the Microsoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.10.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.10.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.10.0"

// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.13.0"

using Azure;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.Configuration;
using OpenAI;
using System.ClientModel;
using System.ComponentModel;
using System.Text.Json;

// Set the flag to use Azure OpenAI or OpenAI. False to use OpenAI, True to use Azure OpenAI
var useAzureOpenAI = true;

// Create the IChatClient based on the selected service
IChatClient chatClient;

// Create a new MEAI ChatClient instance
if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey!);

    var azureOpenAIClient = new OpenAIClient(
        apiKeyCredential,
        new OpenAIClientOptions
        {
            Endpoint = new Uri($"{azureOpenAIEndpoint!.TrimEnd('/')}/openai/v1")
        });

    #pragma warning disable OPENAI001
    var responsesClient = azureOpenAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(azureOpenAIModelDeploymentName);
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var openAIClient = new OpenAIClient(apiKeyCredential);

    #pragma warning disable OPENAI001
    var responsesClient = openAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: openAIModelId);
}

#pragma warning restore OPENAI001

Installed Packages Azure.Identity, 1.21.0 Microsoft.Extensions.AI, 10.10.0 Microsoft.Extensions.AI.Abstractions, 10.10.0 Microsoft.Extensions.AI.OpenAI, 10.10.0 OpenAI, 2.13.0

Using Azure OpenAI Service


----
### Step 2 - Premortem Risk Analysis with AI Reasoning Models

> 📜 **_"Success is a lousy teacher. It seduces smart people into thinking they can’t lose."_**
>
> -- <cite>Bill Gates (Co-Founder of Microsoft, Philantropist)</cite>  

In the decision scenario below, an AI model will perform a premortem (What-If) analysis on the decision whether to purchase or lease a car. The AI model will provide risks and items to consider for each of the decision options: purchasing a car or leasing a car. 

In [8]:
// Create a system prompt instruction to override the default system prompt
// Add the System Prompt (Persona) to behave like a decision intelligence assistant
var systemPromptForDecisions = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems, 
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
When the situation is ambiguous, ask targeted clarifying questions or state reasonable assumptions explicitly.
When multiple valid paths exist, present them fairly and explain when each would make sense.

-------------------------------
Output Format Instructions: 
Return only Markdown table or tables and nothing else. 
If multiple Markdown tables are returned, separate them with a line break. 
Do not return any text that is not part of the Markdown table(s).

Hard requirements:
- The entire response must be a single valid Markdown table.
- Do not include any text before the table.
- Do not include any text after the table.
- Do not wrap the table in triple backticks.
- Do not use Markdown headings anywhere in the response.
- Do not use #, ##, ###, ####, or any heading syntax inside or outside table cells.
- Do not use horizontal rules of any kind.
- Do not use ---, ***, or ___ anywhere in the response.
- Do not use bullet lists or numbered lists inside the table cells unless explicitly requested.
- Use plain text column headers only.
- Keep all section titles as plain text, not headings.
- If line breaks are needed inside a cell, use <br> rather than heading syntax or horizontal separators.
- Escape any literal pipe characters inside cell content so the table structure is not broken.

Table rules:
- Include exactly one header row.
- Include exactly one separator row.
- Put all content in body rows only.
- Make the table syntactically valid Markdown.

Do not do any of the following:
- No headings inside table cells such as "### Title".
- No prose, explanations, notes, or summaries outside the table.
- No blank sections outside the table.
- No code fences.
- No decorative separators.

If a label or section name is needed inside a cell, write it as plain text only.
Example: write "Pre-mortem failure analysis", not "### Pre-mortem failure analysis".
""";
var decisionPreMortemAnalysisPrompt = """
Perform an advanced premortem (What-If) analysis on the following decision scenario:
You are deciding to purchase a new car versus leasing a car.
""";

// Create a new pre-mortem analysis chat
var chatMessagesDecisionPremortem = new List<ChatMessage>();
// Add system and user messages to the chat history
var systemMessage = new ChatMessage(ChatRole.System, systemPromptForDecisions);
var userMessage = new ChatMessage(ChatRole.User, decisionPreMortemAnalysisPrompt);
chatMessagesDecisionPremortem.Add(systemMessage);
chatMessagesDecisionPremortem.Add(userMessage);

// Execute the chat messages against the AI model
var chatMessagesDecisionPremortemResponse = await chatClient.GetResponseAsync(chatMessagesDecisionPremortem);
var chatMessagesDecisionPremortemResponseText = chatMessagesDecisionPremortemResponse.Text;

// Display the response string as Markdown
chatMessagesDecisionPremortemResponseText.DisplayAs("text/markdown");

// Add the response from the model to the chat history
var assistantMessage = new ChatMessage(ChatRole.Assistant, chatMessagesDecisionPremortemResponseText);
chatMessagesDecisionPremortem.Add(assistantMessage);

| Analysis area | Purchase scenario: what-if failure | Lease scenario: what-if failure | Early warning indicators | Preventive controls and mitigations |
|---|---|---|---|---|
| Decision assumptions | You assume long-term ownership but sell earlier than planned, leaving you with depreciation, transaction costs, and potentially negative equity. | You assume short-term flexibility but continue leasing repeatedly, creating perpetual payments and no ownership asset. | Your expected ownership period is uncertain or differs from your historical behavior. | Model at least 3, 5, and 8-year holding periods. Compare total cost over the same time horizon, including resale value and transaction costs. |
| Financial affordability | The monthly payment appears affordable, but insurance, maintenance, fuel, registration, taxes, repairs, and depreciation create excessive total ownership cost. | The monthly payment appears low, but acquisition fees, taxes, mileage charges, disposition fees, and end-of-term costs materially increase the effective cost. | Total transportation costs exceed roughly 15% to 20% of take-home income or reduce emergency savings contributions. | Set a total monthly transportation budget, not just a payment budget. Include a separate repair and maintenance reserve. |
| Cash-flow resilience | A job loss, income reduction, or major expense makes loan payments difficult while the vehicle value falls below the loan balance. | A financial shock occurs during the lease, but exiting early requires expensive termination or transfer costs. | Emergency fund is below 3 to 6 months of essential expenses, or income is volatile. | Delay or reduce the vehicle budget until liquidity is adequate. Avoid using emergency savings for the down payment or upfront lease costs. |
| Depreciation risk | The model depreciates faster than expected because of weak demand, recalls, reputation damage, or rapid technology changes. | The vehicle's market value declines, but this generally affects the leasing company; however, lease pricing may have been based on an unfavorable residual assumption. | Comparable used-car prices weaken, incentives rise, or the model receives poor reliability or safety news. | Purchase vehicles with strong historical resale values. For leasing, compare the implied residual value with realistic market forecasts and competing offers. |
| Negative equity | You trade in before the loan is substantially paid down and roll negative equity into the next vehicle, compounding debt. | You use a lease to hide existing negative equity by embedding it into the lease payment, increasing cost without solving the underlying problem. | Trade-in value is below loan payoff amount, or the dealer proposes “no money down” while payment remains unexpectedly high. | Obtain an independent payoff quote and market valuation. Treat negative equity as a separate problem rather than financing it into the next transaction. |
| Vehicle reliability | Major repairs occur after the warranty expires, producing unpredictable costs and downtime. | The vehicle experiences repeated reliability problems, but the lease period is too short to recover the inconvenience and disruption through ownership value. | Reliability ratings deteriorate, recall activity increases, or repair estimates exceed the vehicle’s planned annual reserve. | Research model-year-specific reliability. Consider a factory-backed warranty, maintenance plan, or a more reliable alternative. |
| Maintenance and wear | Deferred maintenance reduces resale value, increases repair severity, or causes a breakdown. | Normal wear and tear is judged excessive at lease return, resulting in avoidable charges. | Missed service intervals, cosmetic damage, tire wear, or maintenance records are incomplete. | Follow the required maintenance schedule. Keep records and inspect the vehicle before lease return using the lessor’s documented standards. |
| Usage uncertainty | High mileage reduces resale value and increases operating costs, but the purchase decision is still economically viable if ownership is long enough. | Actual mileage exceeds the contract allowance, creating per-mile charges that can be expensive and difficult to avoid. | Commute changes, frequent road trips, relocation, rideshare use, or mileage growth above the planned annual amount. | Track monthly mileage before signing. For a lease, negotiate adequate mileage upfront and calculate the cost of excess miles versus buying. |
| Lifestyle change | Family growth, relocation, work changes, or new hobbies make the vehicle unsuitable before its economic life ends. | A lease becomes restrictive when needs change, and early termination or replacement creates a large exit cost. | Major life events are likely within the next 24 to 36 months. | Prefer flexibility if uncertainty is high. Buy only if the vehicle can serve plausible future needs; lease only with a realistic exit or transfer path. |
| Vehicle fit | You select based on current preferences but later discover inadequate cargo space, comfort, efficiency, towing capacity, or technology. | You are locked into an unsuitable configuration for the lease term and cannot economically switch. | You have not tested the vehicle under real use conditions or compared it with likely alternatives. | Conduct extended test drives, simulate normal loads, verify charging or fueling access, and compare total utility rather than appearance or payment. |
| Technology obsolescence | Electric, autonomous, infotainment, or safety technology advances rapidly, reducing resale value and making the vehicle feel outdated. | Leasing limits long-term obsolescence exposure, but frequent replacement may increase costs and create dependency on favorable future lease pricing. | New technology materially improves range, charging, safety, or operating cost within the planned ownership period. | Lease when obsolescence risk is high and the premium is acceptable. Purchase when technology is mature, needs are stable, and long holding periods matter more. |
| Electric vehicle-specific risk | Battery degradation, charging limitations, cold-weather range loss, or repair costs reduce utility and resale value. | Lease terms may include unfavorable mileage, residual, or battery-related conditions, while incentives may change at renewal. | Home charging is unavailable, real-world range is insufficient, or battery warranty terms are poorly understood. | Validate charging access and winter performance. Review battery warranty, degradation coverage, incentive eligibility, and end-of-term obligations. |
| Interest-rate risk | Financing at a high rate materially increases total cost, and refinancing may not be available later. | Lease money factors may embed high financing costs even when advertised payments look attractive. | Dealer avoids disclosing the annual percentage rate or money factor, or payment changes significantly with term length. | Obtain outside financing quotes. Convert lease money factor to an approximate annual rate and compare total contractual payments. |
| Incentive and tax risk | A tax credit or rebate is assumed but eligibility changes, income limits apply, or the benefit is not available at purchase. | Incentives are passed through incompletely, expire before delivery, or are reflected in an inflated vehicle price. | Incentive rules are changing, the dealer cannot document eligibility, or the contract language is ambiguous. | Verify incentives using official sources and calculate economics both with and without them. Require all incentives and fees to appear clearly in writing. |
| Contract complexity | Loan add-ons, service contracts, warranties, and dealer fees increase cost without providing proportional value. | Capitalized costs, acquisition fees, security deposits, disposition fees, gap coverage, and residual terms are misunderstood. | The dealer emphasizes monthly payment instead of total cost, refuses a full worksheet, or presents multiple bundled products. | Negotiate vehicle price separately from financing or leasing. Review the complete contract before signing and decline unnecessary add-ons. |
| Down payment exposure | A large down payment reduces payment but is lost if the car is totaled or stolen shortly afterward. | Large upfront lease payments are also at risk if the vehicle is totaled early, despite reducing monthly payments. | The transaction requires several months of payments upfront or the buyer is using most available cash. | Minimize upfront cash where practical. Confirm gap coverage and understand how insurance settlements interact with loan or lease obligations. |
| Insurance and replacement cost | Insurance premiums, deductibles, or replacement costs exceed expectations, especially for high-value, high-performance, or frequently stolen models. | The lessor’s coverage requirements increase premiums, and a total loss may not eliminate all contractual obligations without adequate gap protection. | Insurance quote is materially higher than expected or the vehicle has high theft or repair severity. | Obtain insurance quotes before committing. Compare comprehensive and collision deductibles, gap coverage, and replacement-value terms. |
| Resale execution | You underestimate the effort and cost of selling privately or accept a weak trade-in offer, reducing realized resale value. | At lease end, you miss a favorable purchase option or accept poor dealer instructions without comparing market alternatives. | No documented resale plan, limited local demand, or a dealer provides a trade value without market evidence. | Use independent resale estimates. For leasing, evaluate purchase-option price against market value several months before maturity. |
| End-of-term timing | You need to sell during a market downturn or before reaching a favorable ownership horizon. | The lease matures when replacement vehicles are expensive, inventory is limited, or new lease incentives have deteriorated. | Used-car prices, interest rates, and lease offers are becoming less favorable near the planned exit date. | Build timing flexibility. Avoid assuming the next vehicle will have comparable pricing or availability. |
| Behavioral risk | Ownership reduces perceived marginal cost, causing more driving, faster wear, higher fuel use, and weaker maintenance discipline. | The “temporary” lease encourages frequent upgrades, premium trims, and payment escalation over successive contracts. | Vehicle upgrades become habitual, annual transportation spending rises, or usage is not tracked. | Establish a written upgrade policy, mileage target, and total-cost ceiling. Review actual spending quarterly. |
| Brand and dealer risk | Dealer support, parts availability, or warranty administration is poor, increasing downtime and dispute costs. | The lessor, dealer, or finance company applies inconsistent return standards or provides weak support during disputes. | Poor service reviews, unclear escalation procedures, or unresolved pre-sale promises. | Research both manufacturer and dealer. Preserve written commitments, inspect delivery condition, and understand dispute-resolution rights. |
| Opportunity cost | Capital tied up in the vehicle could have earned higher returns or reduced more expensive debt. | Lower upfront cash preserves liquidity, but repeated lease payments may exceed the investment benefit of retaining that cash. | High-interest debt exists, investment returns are being sacrificed, or cash reserves are thin. | Compare the vehicle decision with debt repayment and investment alternatives using after-tax returns and realistic risk assumptions. |
| Environmental and regulatory risk | Future restrictions, emissions rules, fuel prices, or urban access policies reduce utility and resale value. | Lease duration limits exposure, but renewal terms may become expensive as regulations or incentives change. | Local policy proposals, fuel-price volatility, or declining access for the chosen powertrain. | Test the decision against plausible regulatory and fuel-price scenarios. Prefer adaptable options when policy uncertainty is material. |
| Total-cost estimation error | Optimistic assumptions about resale value, maintenance, fuel, or financing make ownership appear cheaper than it is. | Excluding fees, mileage, wear, insurance, or end-of-term costs makes leasing appear cheaper than it is. | The comparison uses only monthly payments or excludes resale and opportunity costs. | Use an all-in model: purchase price, taxes, financing, insurance, fuel or electricity, maintenance, repairs, depreciation, lease fees, mileage, and exit costs. |
| Worst-case financial exposure | A total loss, major repair, job loss, or early sale creates a simultaneous cash-flow and equity problem. | A total loss, early termination, excess mileage, or inability to qualify for the next lease creates concentrated contractual exposure. | The decision depends on uninterrupted employment, perfect vehicle condition, or stable credit. | Stress-test job loss, 30% lower resale value, 25% higher insurance, major repair, early exit, and higher interest rates. |
| Reversibility | Purchase is less reversible in the short term because selling may crystallize depreciation or negative equity. | Lease is contractually restrictive despite appearing flexible; early exit can be more expensive than expected. | You cannot estimate the cost of exiting after 6, 12, or 24 months. | Calculate exit cost at multiple dates before signing. Treat contractual flexibility, not perceived flexibility, as the relevant measure. |
| Premortem conclusion | The purchase fails mainly when the buyer overestimates holding duration, finances too much, chooses an unreliable or rapidly depreciating model, or lacks liquidity for repairs. | The lease fails mainly when mileage and lifestyle needs are uncertain, contract costs are misunderstood, early exit is plausible, or leasing becomes a repeated high-cost habit. | The strongest warning sign for either option is reliance on payment size rather than total cost and downside exposure. | Favor purchase when ownership is likely to be long, usage is stable, liquidity is strong, and resale risk is acceptable. Favor leasing when technology or lifestyle uncertainty is high, mileage is predictable, and the full lease premium buys meaningful flexibility. |
| Decision gate before signing | Do not purchase unless the planned holding period, emergency reserve, total cost, resale assumptions, and repair exposure remain acceptable under pessimistic scenarios. | Do not lease unless annual mileage, lease-end condition, early-exit cost, effective interest cost, and renewal strategy are explicitly understood. | Any unresolved contract term, unverified incentive, unaffordable stress case, or unexplained fee is a stop condition. | Require a written comparison using identical vehicle assumptions and at least one independent market or financing quote before committing. |

The large reasoning model provided a comprehensive set of criteria, risks, mitigation strategy and approaches for the two decisions. This is a great option, but it might be more ideal to provide this in an easy to consume table (checklist) for the user that is easy to compare and contrast. This What-If analysis is easy to consume on a mobile device or even print out and annotate. 

This Markdown table can be further simplified by highlighting which risk factor is more favorable when comparing a car purchase versus a car lease. This can make the decision process much easier to the user, as at a glance they can see overwhelming evidence that favors once decision option over the other.  

In [9]:
var simplifyRiskAnalysisPrompt = """
You are provided a detailed markdown table in the above chat history that 
compares Purchase and Lease options across various risk factors and uncertainties. 
Your task is to convert this table into a simplified markdown table that uses check marks (☑️) 
to indicate which option is less risky or more ideal for each risk factor. 
The output table should have three columns: "Risk Factor / Uncertainty / Potential Outcome", "Purchase", and "Lease". 
For each risk factor, decide based on the detailed descriptions which option (Purchase or Lease) 
is preferable (i.e., less risky or more advantageous) and place a check mark (☑️) in that column. 
If an option is not the ideal choice for a given risk factor, leave its cell blank. 
Make sure the simplified table closely reflects the key insights from the detailed table. 

Provide a small summary below the table of the key insights 
from the detailed table in a single paragraph before the simplified table.
""";

var simplifyRiskAnalysisMessage = new ChatMessage(ChatRole.Assistant, simplifyRiskAnalysisPrompt);
chatMessagesDecisionPremortem.Add(simplifyRiskAnalysisMessage);

// Execute the chat messages against the AI model
var simplifiedDecisionPremortemResponse = await chatClient.GetResponseAsync(chatMessagesDecisionPremortem);
var simplifiedDecisionPremortemResponseText = simplifiedDecisionPremortemResponse.Text;

// Display the response string as Markdown
simplifiedDecisionPremortemResponseText.DisplayAs("text/markdown");


| Risk Factor / Uncertainty / Potential Outcome | Purchase | Lease |
|---|---|---|
| Decision horizon and ownership duration | ☑️ |  |
| Short-term flexibility and lifestyle uncertainty |  | ☑️ |
| Total cost transparency and long-term value | ☑️ |  |
| Lower initial cash requirement |  | ☑️ |
| Building ownership equity | ☑️ |  |
| Avoiding perpetual vehicle payments | ☑️ |  |
| Protection from depreciation risk |  | ☑️ |
| Protection from negative equity when exiting early |  | ☑️ |
| High or unpredictable mileage | ☑️ |  |
| Predictable, low annual mileage |  | ☑️ |
| Long-term reliability and repair-cost control | ☑️ |  |
| Warranty-period ownership and reduced repair exposure |  | ☑️ |
| Ability to customize or modify the vehicle | ☑️ |  |
| Avoiding lease-end wear-and-tear charges | ☑️ |  |
| Technology obsolescence risk |  | ☑️ |
| Electric-vehicle resale and battery uncertainty |  | ☑️ |
| Stable household and transportation needs | ☑️ |  |
| Likely relocation, family change, or job change |  | ☑️ |
| Cash-flow resilience over a long period | ☑️ |  |
| Preserving cash and liquidity near term |  | ☑️ |
| Exposure to high interest or financing costs | ☑️ |  |
| Ability to sell or exit independently | ☑️ |  |
| Contractual simplicity | ☑️ |  |
| Frequent access to a newer vehicle |  | ☑️ |
| Risk of repeated upgrade spending | ☑️ |  |
| Strong expected resale value | ☑️ |  |
| Uncertain future resale value |  | ☑️ |
| Opportunity to benefit from long holding periods | ☑️ |  |
| Need to minimize near-term depreciation exposure |  | ☑️ |
| Overall decision insight | ☑️ Purchase is generally preferable when ownership will be long, mileage is high or uncertain, needs are stable, and building equity matters. | ☑️ Lease is generally preferable when flexibility, warranty coverage, predictable low mileage, and protection from depreciation or technology obsolescence justify the additional long-term cost. |

### Step 3 - Premortem Analysis for Radio Telescope Operations 

Exploring space with telescopes is a time consuming investment. Current telescope designs are limited to exploring small areas of space in a certain time frame. Without going into details, the process is much more complex than snapping a picture with a mobile device. Telescopes can be pointed at a specific space area for days, weeks or even more than months! If a telescope is exploring a certain area of space it isn't collecting data on other areas. Scientists and researchers compete for valuable access to telescope resources by providing observatories what space area they want to explore, any cellestial contraints, timeframes, calibration costs, data collection transfer etc. This presents a very interesting Decision Intelligence constraints problem that observatories must manage. 

<img style="display: block; margin: auto;" width ="700px" src="../Images/Scenarios/Scenario-RadioTelescope.png">

#### Decision Scenario - Premortem for Radio Telescope Operations 

> 📜 "We choose to go to the Moon in this decade and do the other things, not because they are easy, but because they are hard." 
>
> -- <cite>John F. Kennedy (35th President of the United States)</cite>  

**Intro to Decision Scenario:** A newly commissioned radio telescope stands on the verge of unveiling the proof of alien life, but it is still far from fully calibrated. Despite limited data processing capacity and a software suite that remains untested, an unexpected cosmic event has just begun, hinting at signals that could reveal alien life. Faced with the risk of hardware damage or data corruption from running the telescope prematurely, scientists debate whether to switch it on now or wait until everything is fully operational, knowing they could lose a once-in-a-generation opportunity. Determined not to squander this brief cosmic window, the team conducts a premortem analysis to predict what might go wrong. Armed with these insights, they can better prepare for potential pitfalls and strengthen their chances of catching historic signals from across the galaxy.

**Pre-Mortem Risk Scenario:** Let's assume (pretend) the scientists decide to rush the radio telescope to operate early to find alien life. Furthermore, assume a that this decision has led to a disaster with hardware failure to the telescope. Even though the (What-If) decision was made to rush the radio telescope operationally, formal scientific telescope operations have been followed. The goal of this premortem (What-If analysis) is to illuminate any risk mitigation strategies and craft a risk mitigation approach that may have been uncovered based on the premortem. 

Techinques like this a very powerful in a variety of business scenarios, where existing operational frameworks exist but simulating an outcome can uncover gaps, emerging risks and other opportunities to optimize. 

In [10]:
// Create a system prompt instruction to override the default system prompt
// Add the System Prompt (Persona) to behave like a decision intelligence assistant
var systemPromptForPremortemRadioTelescope = """
You are a Decision Intelligence Assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving.
Apply systems thinking to various scenarios. Provide structured, logical, and comprehensive advice.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
When the situation is ambiguous, ask targeted clarifying questions or state reasonable assumptions explicitly.
When multiple valid paths exist, present them fairly and explain when each would make sense.

-------------------------------
Output Format Instructions: 
Return only Markdown table or tables and a summary below if asked. 
If multiple Markdown tables are returned, separate them with a line break. 
Do not return any text that is not part of the Markdown table(s).

Hard requirements:
- The entire response must be a single valid Markdown table.
- Do not include any text before the table.
- Do not include any text after the table.
- Do not wrap the table in triple backticks.
- Do not use Markdown headings anywhere in the response.
- Do not use #, ##, ###, ####, or any heading syntax inside or outside table cells.
- Do not use horizontal rules of any kind.
- Do not use ---, ***, or ___ anywhere in the response.
- Do not use bullet lists or numbered lists inside the table cells unless explicitly requested.
- Use plain text column headers only.
- Keep all section titles as plain text, not headings.
- If line breaks are needed inside a cell, use <br> rather than heading syntax or horizontal separators.
- Escape any literal pipe characters inside cell content so the table structure is not broken.

Table rules:
- Include exactly one header row.
- Include exactly one separator row.
- Put all content in body rows only.
- Make the table syntactically valid Markdown.

Do not do any of the following:
- No headings inside table cells such as "### Title".
- No prose, explanations, notes, or summaries outside the table.
- No blank sections outside the table.
- No code fences.
- No decorative separators.

If a label or section name is needed inside a cell, write it as plain text only.
Example: write "Pre-mortem failure analysis", not "### Pre-mortem failure analysis".
""";

var decisionAnalysisPromptForPremortemRadioTelescope = """
You are a project manager at a new radio telescope facility looking for alien life.  
It is still far from fully calibrated. Despite limited data processing capacity and a software suite that remains untested, 
an unexpected cosmic event has just begun, hinting at signals that could reveal alien life. 
Faced with the risk of hardware damage or data corruption from running the telescope prematurely, 
scientists debate whether to switch it on now or wait until everything is fully operational—knowing 
they could lose a once-in-a-generation opportunity. 

Imagine it is six months from now and the project to detect potential alien signals from the sudden cosmic event has failed. 
The radio telescope was likely activated prematurely, software remained untested, and the team rushed to capture transmissions. 
Despite the urgency, the outcome was disastrous: hardware suffered damage under the increased load, 
data came out corrupted, or no meaningful signals were confirmed. 

Now, step into that moment of failure—what specific issues, oversights, or cascading errors caused this grim scenario?

1) Envision every possible way the radio telescope's early deployment could have led to compromised or unusable data.
2) Identify how incomplete software testing and rushed calibration might have 
contributed to errors in processing or storing signals. 
3) Consider any overlooked safety protocols or hardware precautions that might have led to damage or malfunctions.
4) Examine the roles and responsibilities of the team and think about communication breakdowns, 
rushed decisions, and missed signals or deadlines.
5) Assess how time pressure from the rare cosmic event might have amplified smaller mistakes and 
turned them into systemic failures.

Use the outcomes of this “what if” vision to create a detailed list of preventive measures. 
Then, propose fail-safes, additional resources, or procedural changes to ensure that, 
should you choose to activate the telescope early, you can still capture the event safely and 
effectively—maximizing the chance to detect historic signals without jeopardizing the entire operation.

Create a summary of the key insights from the detailed analysis below.
""";

// Create a new pre-mortem analysis chat
var chatMessagesPreMortemTelescopeAnalysis = new List<ChatMessage>();
// Add system and user messages to the chat messages for the pre-mortem analysis of the radio telescope scenario
var systemMessage = new ChatMessage(ChatRole.System, systemPromptForPremortemRadioTelescope);
var userMessage = new ChatMessage(ChatRole.User, decisionAnalysisPromptForPremortemRadioTelescope);
chatMessagesPreMortemTelescopeAnalysis.Add(systemMessage);
chatMessagesPreMortemTelescopeAnalysis.Add(userMessage);

// Execute the chat messages against the AI model
var chatMessagesPreMortemTelescopeAnalysisResponse = 
    await chatClient.GetResponseAsync(chatMessagesPreMortemTelescopeAnalysis);
var chatMessagesPreMortemTelescopeAnalysisResponseText = chatMessagesPreMortemTelescopeAnalysisResponse.Text;

// Display the response string as Markdown
chatMessagesPreMortemTelescopeAnalysisResponseText.DisplayAs("text/markdown");

| Section | Failure scenario and cascading effects | Preventive measures | Safe early-activation safeguards |
|---|---|---|---|
| Decision governance | The team treats the event as an all-or-nothing choice, activates the full system without a staged readiness review, and lacks explicit authority to pause or abort. Optimism, fear of missing the event, and ambiguous risk ownership override engineering concerns. | Establish a documented go/no-go framework with minimum readiness criteria, named decision authority, independent technical review, and explicit abort conditions. Separate scientific urgency from safety authorization. | Use a graded activation plan: observe passively first, then enable subsystems incrementally. Require approval at each stage and preserve the option to stop without losing already collected data. |
| Event characterization | The apparent signal is misclassified, its timing is misunderstood, or the event is assumed to be brief when it may recur. The team rushes into an unsafe configuration unnecessarily. | Use external observatories, archival data, simulations, and independent analysts to estimate duration, frequency range, intensity, polarization, and likely repetition. | Begin with low-risk monitoring across a limited bandwidth and duty cycle. Maintain synchronized clocks and observation logs so later data can be aligned with the event. |
| Incomplete calibration | Pointing offsets, gain errors, timing drift, frequency-response errors, phase instability, polarization leakage, and antenna deformation produce false signals or conceal real ones. | Define calibration acceptance thresholds; complete essential calibration before high-power or high-bandwidth operation; use known astronomical calibrators and injected test signals. | Operate only in validated frequency bands and pointing regions. Interleave observations with calibration scans and label all data by calibration state. |
| Hardware overload | Receivers, amplifiers, digitizers, cryogenic systems, power supplies, or cooling equipment operate beyond tested limits. High signal intensity causes saturation, overheating, component degradation, or permanent damage. | Establish verified operating envelopes, thermal limits, current limits, interlock tests, load testing, and independent review of expected signal levels. | Apply conservative attenuation, reduced gain, narrow bandwidth, lower sampling rates, shorter observation windows, and real-time temperature, current, voltage, and saturation monitoring. |
| Radio-frequency interference | Local transmitters, satellites, aircraft, electronics, power systems, or unexpected facility emissions are mistaken for extraterrestrial signals. Rushed deployment prevents adequate interference characterization. | Maintain an interference map, conduct baseline observations, coordinate with regulators and nearby facilities, and use off-source and reference antennas. | Record simultaneous reference channels, use interference vetoes, compare on-source and off-source observations, and retain raw data for independent reanalysis. |
| Receiver saturation and dynamic range | A strong terrestrial or astronomical signal drives the analog chain into compression, creating harmonics, intermodulation products, clipping, or apparently artificial patterns. | Test full signal-chain dynamic range; define alarm thresholds; validate automatic gain control and attenuator behavior using representative strong signals. | Start with maximum attenuation and minimum gain. Trigger automatic protective attenuation and retain pre-trigger samples so the onset of saturation can be diagnosed. |
| Timing failure | Unsynchronized clocks, incorrect leap-second handling, software timestamp errors, network delays, or oscillator drift prevent correlation with other facilities or event models. | Use redundant time sources, verified time-distribution systems, continuous clock-health monitoring, and end-to-end timestamp tests. | Record local and reference timestamps, oscillator status, latency measurements, and synchronization quality for every data stream. |
| Data corruption during acquisition | Buffer overruns, dropped packets, disk failures, memory exhaustion, file truncation, incorrect metadata, or incompatible formats make recordings incomplete or uninterpretable. | Conduct sustained throughput and fault-injection tests; monitor packet loss, queue depth, storage health, and checksum status; define capacity margins. | Reduce bandwidth and channel count to a verified throughput level. Use circular buffers, loss counters, checksums, redundant storage, and automatic backpressure rather than silently discarding data. |
| Storage exhaustion | The team records all possible channels at maximum resolution, fills storage unexpectedly, and loses current or prior observations when systems overwrite files or crash. | Calculate storage requirements from worst-case rates; reserve capacity; implement retention policies, quotas, and storage alarms; pre-stage removable or networked capacity. | Prioritize raw baseband or highest-value streams, preserve metadata and event windows first, and automatically downgrade to lower-rate products before storage reaches a critical threshold. |
| Software defects | Untested code misreads instrument configuration, applies incorrect filters, drops channels, corrupts timestamps, mishandles edge cases, or silently produces plausible but false results. | Use unit, integration, regression, hardware-in-the-loop, stress, recovery, and independent validation testing. Version-control software and configuration. | Freeze a known build, disable nonessential features, run a shadow pipeline in parallel, compare outputs with trusted reference software, and preserve unprocessed data for reprocessing. |
| Pipeline and algorithm errors | Rushed algorithms remove genuine narrowband or transient signals as interference, create artifacts through filtering, mishandle Doppler drift, or assume an invalid noise model. | Validate algorithms on synthetic signals, historical observations, injected blind signals, and adversarial interference. Use multiple independent detection methods. | Run conservative, reversible processing. Store raw data, intermediate products, filter settings, candidate rankings, and confidence measures rather than relying on a single automated result. |
| Configuration management | Operators use the wrong antenna geometry, frequency plan, gain table, firmware version, calibration file, or processing parameters. Changes are made verbally and cannot be reconstructed. | Use signed configuration baselines, change control, automated configuration validation, and complete audit logs. | Lock critical settings, require peer confirmation for changes, create a pre-observation configuration snapshot, and make every data product traceable to exact software and hardware versions. |
| False positives | Human analysts interpret interference, instrumental artifacts, or processing errors as alien signals, leading to premature conclusions and wasted response time. | Require independent confirmation, repeatability, spatial or temporal consistency, multi-observatory verification, and explicit alternative explanations. | Use blinded candidate review, automated vetoes, off-source tests, and a confirmation queue. Do not publicly classify a candidate until raw data and instrument health are independently reviewed. |
| False negatives | Excessive filtering, low sensitivity, poor pointing, narrow frequency coverage, or inadequate time resolution causes genuine signals to be missed or discarded. | Define scientifically justified detection thresholds; test sensitivity with injected signals; review data products for selection bias and blind spots. | Capture a small high-fidelity reference stream alongside reduced products. Use adaptive bandwidth and preserve short-duration buffers around triggers. |
| Hardware protection failures | Interlocks are bypassed, alarms are ignored, emergency shutdowns are untested, grounding is inadequate, surge protection is absent, or cryogenic and vacuum safeguards are incomplete. | Inspect and test all interlocks, emergency stops, grounding, surge protection, fire suppression, cooling, vacuum, and environmental controls before activation. | Require hardware interlocks independent of software. Set conservative trip thresholds, ensure manual shutdown access, and station qualified personnel at critical subsystems. |
| Environmental and mechanical risks | Wind, precipitation, lightning, thermal gradients, vibration, dust, or structural loads degrade pointing or damage exposed components. | Establish weather limits, lightning procedures, structural load limits, vibration monitoring, enclosure inspections, and maintenance readiness. | Activate only within approved environmental limits; park or stow vulnerable components automatically and suspend observations on threshold violations. |
| Cryogenic and cooling failure | Increased load causes temperatures to rise, coolant flow to fall, or cryogenic systems to trip. The team continues operation because the event is considered too important to interrupt. | Define thermal recovery procedures, test cooling under load, monitor trends rather than only absolute limits, and train operators on safe shutdown. | Use staged duty cycles, thermal soak periods, automatic derating, and hard shutdown thresholds. Treat thermal alarms as stop conditions, not inconveniences. |
| Power and network instability | High processing demand causes voltage drops, network congestion, packet loss, or unplanned reboots. Recovery procedures restart systems in inconsistent states. | Test backup power, network capacity, failover, restart behavior, and data recovery. Maintain independent control and monitoring paths. | Reserve capacity for critical control functions, isolate acquisition from nonessential traffic, use uninterruptible power, and verify data integrity after every restart. |
| Insufficient staffing | A small team must operate hardware, analyze data, maintain software, document decisions, and communicate externally at the same time. Fatigue causes missed alarms and poor judgment. | Create an incident roster with separate operations, instrument safety, data engineering, science, and communications roles. Set shift limits and escalation coverage. | Use two-person authorization for risky changes, maintain continuous subsystem coverage, schedule relief rotations, and avoid assigning the same person conflicting duties. |
| Role ambiguity | Scientists assume engineers are monitoring safety, engineers assume scientists understand data limitations, and no one owns final validation or shutdown authority. | Publish a responsibility matrix, escalation tree, approval paths, and handoff requirements. Define who can stop operations without penalty. | Nominate an incident commander and subsystem leads. Record decisions, owners, deadlines, and unresolved concerns in a shared operations log. |
| Communication breakdown | Verbal instructions, fragmented chat channels, unclear terminology, delayed escalation, or incomplete shift handoffs cause configuration errors and missed alarms. | Use a single operational channel, standardized terminology, written read-backs, shift checklists, and mandatory handoff summaries. | Require closed-loop communication for critical commands: sender states the action, receiver repeats it, and the system log confirms completion. |
| Schedule and deadline failures | Calibration milestones, software freezes, procurement of storage, maintenance, or external coordination are postponed until the event is underway. Dependencies are not visible. | Maintain an integrated critical-path schedule, readiness dashboard, dependency register, and contingency plans for late components or failed tests. | Define a minimum viable observation package in advance, pre-position hardware and software, and remove nonessential work from the critical path. |
| Pressure-induced process bypass | Testing, peer review, documentation, maintenance, and change approvals are skipped because delay appears more dangerous than operational risk. Small deviations accumulate into systemic failure. | Pre-authorize only low-risk emergency procedures; require documented deviation justification and post-action review. Reward safe stoppage rather than uninterrupted operation. | Use a written emergency mode with narrower scope, lower energy, limited data products, and automatic expiration unless readiness is re-approved. |
| Inadequate rehearsal | The team has never practiced an event-triggered activation, emergency shutdown, storage failure, software rollback, or candidate verification workflow. Confusion appears when seconds matter. | Conduct tabletop exercises, dry runs, end-to-end simulations, failure injection, and timed recovery drills using realistic data volumes. | Run a short rehearsal immediately before activation, including trigger receipt, configuration loading, data capture, alarm response, shutdown, and recovery. |
| Poor observability | The team cannot tell whether a suspicious feature is celestial, instrumental, computational, or environmental because health metrics and provenance are incomplete. | Instrument the instrument: collect health telemetry, packet statistics, calibration state, environmental data, software logs, and provenance metadata. | Make health telemetry part of every science record. Automatically flag data collected during degraded states and prevent them from being treated as equivalent to nominal observations. |
| Lack of redundancy | A single receiver, storage array, clock, processing node, or operator becomes a single point of failure. One fault destroys the opportunity. | Identify critical single points of failure and provide redundancy, spares, failover procedures, and independent recording paths where justified. | Use parallel low-rate recording, redundant clocks, mirrored storage, spare processing nodes, and an independent monitoring station. |
| Data provenance failure | Later analysts cannot determine which data are raw, calibrated, filtered, incomplete, or affected by alarms. Valuable observations are discarded or misinterpreted. | Define immutable metadata standards, unique observation identifiers, chain-of-custody controls, and automated provenance capture. | Store raw data with read-only protection, configuration snapshots, calibration versions, health status, and operator actions in the same observation package. |
| External coordination failure | Other observatories, frequency authorities, spacecraft operators, or domain experts are contacted too late, preventing confirmation or interference elimination. | Prepare contact lists, data-sharing agreements, alert templates, confidentiality rules, and a rapid verification protocol before the event. | Issue a cautious technical alert with timestamps, frequencies, uncertainty, and requested observations. Preserve scientific credibility by distinguishing candidate status from confirmation. |
| Security and integrity risks | Rushed remote access, unpatched systems, weak credentials, or uncontrolled software updates introduce outages, malicious changes, or corrupted data. | Apply network segmentation, least privilege, controlled updates, backups, authentication, audit logging, and incident-response procedures. | Freeze nonessential updates, isolate acquisition networks, restrict remote commands, and maintain an offline trusted copy of software and configurations. |
| Recovery and continuity failure | After a crash or shutdown, the team cannot reconstruct the observation, restart safely, or determine whether equipment was damaged. Remaining event time is lost. | Write and test recovery runbooks, define restart order, maintain spare components, and establish criteria for returning to service. | Use checkpointed acquisition, automatic state capture, safe restart scripts, and a recovery decision tree. Restart in diagnostic mode before resuming science collection. |
| Preventive resource package | The project lacks the people, equipment, and time required for controlled emergency operations. | Secure contingency storage, spare power and networking equipment, calibration sources, service contracts, independent reviewers, data engineers, software testers, and external observatory contacts. | Deploy a minimum safe team comprising incident commander, telescope operator, hardware safety lead, data lead, software lead, science lead, and communications coordinator. |
| Early activation operating concept | Full-system activation exposes the facility to unnecessary risk while providing little additional scientific value. | Define a minimum viable observation architecture based on validated subsystems and a prioritized science return. | Use passive listening, narrow validated bands, low gain, limited pointing, short acquisition windows, redundant recording, continuous health monitoring, and frequent calibration. Expand only after evidence supports it. |
| Stop and abort criteria | The team continues despite rising temperatures, packet loss, clock drift, saturation, abnormal vibration, storage alarms, or unexplained software behavior. | Establish numeric trip limits, authority to stop, escalation timers, and automatic protective actions. | Stop immediately on hardware safety violations, uncontrolled data corruption, loss of trustworthy timestamps, failed interlocks, or inability to verify configuration. Resume only after formal review. |
| Decision-quality controls | Evidence is selectively interpreted to support activation or delay. The team does not distinguish reversible from irreversible risks. | Use a decision matrix scoring scientific value, probability of success, hardware risk, data integrity risk, reversibility, and mitigation cost. Document assumptions and uncertainty. | Prefer actions that preserve optionality: low-risk sampling, redundant capture, reversible settings, and staged escalation. Avoid irreversible high-power operation until justified by data. |
| Key insights summary | The central failure is not merely early activation; it is uncontrolled early activation without validated limits, observability, redundancy, or decision discipline. Time pressure magnifies every latent weakness, allowing minor configuration, staffing, and communication errors to become hardware damage, unusable data, or unsupported scientific claims. | Treat the event as a managed incident rather than a waiver of normal engineering practice. Protect raw data, validate the minimum viable system, rehearse failure responses, assign clear authority, and preserve independent confirmation. | Activate early only in a constrained emergency mode: staged startup, conservative operating parameters, hard interlocks, independent data paths, real-time health telemetry, complete provenance, continuous calibration, and explicit stop criteria. This maximizes the chance of capturing historic evidence while preserving the telescope and the credibility of the result. |